In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import numpy as np
import sequence.utils.log as log
from sequence.app.teleport_app import TeleportApp
from sequence.topology.node import DQCNode
from sequence.topology.dqc_net_topo import DQCNetTopo
from sequence.kernel.quantum_utils import random_state, verify_same_state_vector
from sequence.constants import MILLISECOND
from sequence.topology.router_net_topo import RouterNetTopo

from sequence.app.teleport_app import TeleportApp
from sequence.app.relay_teleport_app import RelayTeleportApp

In [3]:
from enum import Enum
from sequence.message import Message

class TestMsgType(Enum):
    TEST = 1

class TestMessage(Message):
    def __init__(self, bit1, bit2,destination):
        super().__init__(TestMsgType.TEST, "test")
        self.bit1 = bit1
        self.bit2 = bit2
        self.destination=destination

    def __str__(self):
        return f"TestMessage(bit1={self.bit1}, bit2={self.bit2})"

In [4]:
def teleport(psi: np.array, seed: int = 0) -> np.array:
    """Do a teleportation
    
    Args:
        psi (np.array): the state to be teleported at Alice
        seed (int): the random seed
    Returns:
        np.array: the state received at Bob
    """
    # read the json config file, initialize the topology
    topo = RouterNetTopo("teleport_3node.json")

    tl   = topo.tl  # timeline

    # set up log
    log_filename = 'teleport_2node.log'
    log.set_logger(__name__, tl, log_filename)
    log.set_logger_level('INFO')
    modules = ['generation', 'teleportation', 'teleport_app']
    for module in modules:
        log.track_module(module)

    # get alice and bob node
    nodes = topo.get_nodes_by_type(RouterNetTopo.QUANTUM_ROUTER)

    alice = next(n for n in nodes if n.name == "alice")
    relay = next(n for n in nodes if n.name == "relay")
    bob = next(n for n in nodes if n.name == "bob")
    # 0) set seed for the bsm node
    # bsm_nodes = topo.nodes.get(DQCNetTopo.BSM_NODE, [])
    # for bsm_node in bsm_nodes:
    #     bsm_node.set_seed(seed)
    
    # 1) Prepare |ψ> in Alice’s data memory
    # data_memo_arr = alice.get_component_by_name(alice.data_memo_arr_name)
    # data_memo_arr[0].update_state(psi)

    # 2) Attach the TeleportApp on both node
    alice_app = TeleportApp(alice)
    relay_app = RelayTeleportApp(relay, next_hop=bob.name)
    bob_app   = TeleportApp(bob)
  

    data_memory_array = alice.owner.get_component_by_name(
        alice.owner.data_memo_arr_name
    )
    data_memory_array.memories[0].update_state(psi)

    print("\n===== INPUT STATE =====")
    print("Alice input psi =", psi)
    print("=======================\n")
    print("Alice DataMemoryArray:", data_memory_array)
    print("Alice data memories:", data_memory_array.memories)

    # 3) Kick off teleport
    alice_app.start(
        responder   = bob.name,
        start_t     = 2  * MILLISECOND,
        end_t       = 1000 * MILLISECOND,
        memory_size = 1,
        fidelity    = 0.6,
        data_memory_index = 0
    )

#     # 3) Configure and start the responder app on Bob FIRST
#     bob_app.start(
#         responder = relay.name,
#         start_t   = 1 * MILLISECOND,
#         end_t     = 500 * MILLISECOND,
#         memory_size = 1,
#         fidelity    = 0.8,
#         data_memory_index = 0
#     )
    
#     relay_app.start(
#         responder=bob.name,
#         start_t=1 * MILLISECOND,
#         end_t=100 * MILLISECOND,
#         memory_size=1,
#         fidelity=0.8,
#         data_memory_index=0
# )
#     print("Alice :", type(alice.teleport_app).__name__)
#     print("Relay :", type(relay.teleport_app).__name__)
#     print("Bob   :", type(bob.teleport_app).__name__)
    
    # 4) Run the simulation

    tl.init()

    print("\n===== CHANNEL CHECK =====")
    print("Alice channels:", list(alice.cchannels.keys()))
    print("Relay channels:", list(relay.cchannels.keys()))
    print("Bob channels  :", list(bob.cchannels.keys()))
    print("=========================\n")

    # alice.reserve_net_resource(
    #     responder="bob",
    #     start_time=2*MILLISECOND,
    #     end_time=1000*MILLISECOND,
    #     memory_size=1,
    #     target_fidelity=0.8
    # )
    # msg = TestMessage(1, 0,bob.name)
    # forwading=alice.network_manager.protocol_stack[0]
    # forwading.push(dst="bob",msg=msg)

    # print("Alice sending test message...")
    # print(alice.network_manager.protocol_stack)
    print("\n===== FORWARDING TABLES =====")
    print("Alice :", alice.network_manager.get_forwarding_table())
    print("Relay :", relay.network_manager.get_forwarding_table())
    print("Bob   :", bob.network_manager.get_forwarding_table())
    print("============================\n")
    
    # alice.network_manager.protocol_stack[0].push(dst="bob", msg=msg)
    tl.run()

    print("\n===== FINAL RESULT =====")

    print("Alice input :", psi)
    print("Bob output  :", bob_app.results)

    return 

    # 5) simulation finished, read out Bob’s communication qubit state
    # print("Bob results =", bob_app.results)
    # print("Relay results =", relay_app.results)
    # print("Alice results =", alice_app.results)

    # return alice_app.results, relay_app.results, bob_app.results
    # return np.array(teleported_qubit)

In [5]:
def main():

    # psi = np.array(random_state())
    seed = np.random.randint(0, 1000)
    
    # teleported_psi = teleport(psi, seed)  # run the teleporation
    # psi = np.array([1 / np.sqrt(2), 1 / np.sqrt(2)])
    psi = np.array([
        1 / np.sqrt(2),
        1 / np.sqrt(2)
    ], dtype=complex)

    teleport(psi, seed)
#     print(f'psi            = {psi}')
#     print(f'teleported psi = {teleported_psi}')
    
#     assert verify_same_state_vector(teleported_psi, psi), f"teleported state {teleported_psi} != original {psi}"
#     print("Same state =", verify_same_state_vector(teleported_psi, psi))
#     print('\npsi = teleported_psi --> teleporation success!')
    
# ''' print('\nThe teleport_3node.log:\n')'''
# '''with open("Teleportion_3node.log", "r") as f:
#         logs = f.read()
        # print(logs)'''

In [6]:
main()

Routing Type :  routing_xy
Routing Protocol :  XYRoutingProtocol
Routing Type :  routing_xy
Routing Protocol :  XYRoutingProtocol
Routing Type :  routing_xy
Routing Protocol :  XYRoutingProtocol

===== INPUT STATE =====
Alice input psi = [0.70710678+0.j 0.70710678+0.j]

Alice DataMemoryArray: alice.DataMemoryArray
Alice data memories: [<sequence.components.memory.Memory object at 0x0000021E8420B9D0>]
Table      : {}
Message    : |type=RSVPMsgType.REQUEST; reservation=|identity=0; initiator=alice; responder=bob; path=[]; start_time=2,000,000,000; end_time=1,000,000,000,000; target_fidelity=0.6; entanglement_number=1; memory_size=1||
[ROUTING] alice neighbors: ['relay']
[ROUTING] relay neighbors: ['alice', 'bob']
[ROUTING] Path: alice -> relay -> bob

===== TOPOLOGY ROUTING =====
Routing Protocol : XYRoutingProtocol
Current Node     : alice
Destination      : bob
Next Hop         : relay
Forwarding alice -> relay
Future time = 500000000
Receiver    = relay
Process     = <sequence.kernel.